# Module 1 — Information Retrieval
**Dataset**: SQuAD 2.0 (HuggingFace `datasets`)  
**Model**: Hybrid retriever — TF-IDF + BM25 with min-max normalized score fusion  
**Evaluation**: Recall@10, MRR on 500 answerable validation samples

## 1. Install & Imports

In [1]:
!pip install rank_bm25 datasets -q

import os, json, pickle, re, string
import numpy as np
import scipy.sparse as sp
from tqdm import tqdm
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

print('All imports successful!')

All imports successful!


## 2. Load SQuAD 2.0 Dataset

In [2]:
print('Loading SQuAD 2.0 via HuggingFace datasets...')
squad = load_dataset('rajpurkar/squad_v2')

# Build corpus: unique passages from train + validation, deduplicated on context text
seen = set()
corpus_passages = []
context_to_id = {}

for split in ['train', 'validation']:
    for item in squad[split]:          # ← fixed indentation
        ctx = item['context'].strip()
        if ctx not in seen:
            seen.add(ctx)
            context_to_id[ctx] = len(corpus_passages)
            corpus_passages.append(ctx)

print(f'Corpus size      : {len(corpus_passages):,} unique passages')
print(f'Train examples   : {len(squad["train"]):,}')
print(f'Val examples     : {len(squad["validation"]):,}')

Loading SQuAD 2.0 via HuggingFace datasets...


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Corpus size      : 20,233 unique passages
Train examples   : 130,319
Val examples     : 11,873


In [3]:
# Build answerable evaluation samples from the validation split
eval_samples = []
for item in squad['validation']:
    if len(item['answers']['text']) == 0:
        continue   # skip unanswerable
    ctx = item['context'].strip()
    if ctx not in context_to_id:
        continue   # passage not in train corpus
    eval_samples.append({
        'question'   : item['question'].strip(),
        'context_id' : context_to_id[ctx],
    })

print(f'Answerable eval samples : {len(eval_samples):,}')
print(f'Example question        : {eval_samples[0]["question"]}')

Answerable eval samples : 5,928
Example question        : In what country is Normandy located?


## 3. Text Preprocessing

In [4]:
def preprocess(text):
    """Lowercase, remove punctuation, collapse whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    """Simple whitespace tokenizer after preprocessing."""
    return preprocess(text).split()

print(tokenize("What is the capital of France?"))

['what', 'is', 'the', 'capital', 'of', 'france']


## 4. Build TF-IDF Index

In [5]:
print('Building TF-IDF index over corpus...')

tfidf_vectorizer = TfidfVectorizer(
    max_features = 50_000,
    ngram_range  = (1, 2),
    sublinear_tf = True,
)

preprocessed_corpus = [preprocess(p) for p in corpus_passages]
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_corpus)

print(f'TF-IDF matrix shape : {tfidf_matrix.shape}')
print(f'Vocabulary size     : {len(tfidf_vectorizer.vocabulary_):,}')

Building TF-IDF index over corpus...
TF-IDF matrix shape : (20233, 50000)
Vocabulary size     : 50,000


## 5. Build BM25 Index

In [6]:
print('Tokenizing corpus for BM25 (may take ~1-2 min)...')
tokenized_corpus = [tokenize(doc) for doc in tqdm(corpus_passages)]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print('BM25 index built!')

Tokenizing corpus for BM25 (may take ~1-2 min)...


100%|██████████| 20233/20233 [00:02<00:00, 9066.71it/s]


BM25 index built!


## 6. Hybrid Retrieval Function

In [7]:
def retrieve_hybrid(question, top_k=10, alpha=0.5):
    """
    Hybrid retriever: min-max normalized TF-IDF + BM25 score fusion.
    alpha * norm(tfidf_scores) + (1-alpha) * norm(bm25_scores)
    Returns list of (passage_index, score) sorted descending.
    """
    # TF-IDF scores
    q_vec = tfidf_vectorizer.transform([preprocess(question)])
    tf_scores = cosine_similarity(q_vec, tfidf_matrix).flatten()

    # BM25 scores
    bm_scores = np.array(bm25.get_scores(tokenize(question)))

    # Min-max normalization
    def norm(a):
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn + 1e-9)

    combined = alpha * norm(tf_scores) + (1 - alpha) * norm(bm_scores)
    top_idx  = np.argsort(combined)[::-1][:top_k]
    return [(int(i), float(combined[i])) for i in top_idx]

# Quick sanity check
q = "Who invented the telephone?"
print(f'Query: {q}')
for rank, (idx, score) in enumerate(retrieve_hybrid(q, top_k=3), 1):
    print(f'  Rank {rank} | Score {score:.4f} | {corpus_passages[idx][:100]}...')

Query: Who invented the telephone?
  Rank 1 | Score 0.9136 | The Bell Telephone Company was created in 1877, and by 1886, more than 150,000 people in the U.S. ow...
  Rank 2 | Score 0.8056 | In 1948, the point-contact transistor was independently invented by German physicists Herbert Mataré...
  Rank 3 | Score 0.7941 | Bell and his assistant Charles Sumner Tainter jointly invented a wireless telephone, named a photoph...


## 7. Evaluation — Recall@10 and MRR

In [8]:
def evaluate_ir(retriever_fn, eval_set, top_k=10):
    """
    Evaluate IR retriever. Only answerable questions are used.
    Prints and returns Recall@k, MRR, and n.
    """
    recall_hits      = 0
    reciprocal_ranks = []

    for sample in tqdm(eval_set, desc='Evaluating'):
        gold_id       = sample['context_id']
        results       = retriever_fn(sample['question'], top_k=top_k)
        retrieved_ids = [r[0] for r in results]

        if gold_id in retrieved_ids:
            recall_hits += 1
            rank = retrieved_ids.index(gold_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    return {
        'Recall@10' : recall_hits / len(eval_set),
        'MRR'       : float(np.mean(reciprocal_ranks)),
        'n'         : len(eval_set),
    }

# Evaluate hybrid retriever on 500 answerable validation samples
print('Evaluating Hybrid Retriever (TF-IDF + BM25)...')
results = evaluate_ir(retrieve_hybrid, eval_samples[:500], top_k=10)

print(f'\nRecall@10 : {results["Recall@10"]:.4f}')
print(f'MRR       : {results["MRR"]:.4f}')
print(f'n         : {results["n"]}')

Evaluating Hybrid Retriever (TF-IDF + BM25)...


Evaluating: 100%|██████████| 500/500 [01:29<00:00,  5.60it/s]


Recall@10 : 0.8460
MRR       : 0.6693
n         : 500


## 8. Save Artifacts

In [9]:
os.makedirs('/kaggle/working/ir_artifacts', exist_ok=True)

# 1. TF-IDF vectorizer
with open('/kaggle/working/ir_artifacts/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# 2. TF-IDF sparse matrix
sp.save_npz('/kaggle/working/ir_artifacts/tfidf_matrix.npz', tfidf_matrix)

# 3. BM25 index
with open('/kaggle/working/ir_artifacts/bm25.pkl', 'wb') as f:
    pickle.dump(bm25, f)

# 4. Corpus index
with open('/kaggle/working/ir_artifacts/corpus_index.json', 'w') as f:
    json.dump({'ids': list(range(len(corpus_passages))),
               'passages': corpus_passages}, f)

print('Artifacts saved to /kaggle/working/ir_artifacts/:')
print('  tfidf_vectorizer.pkl')
print('  tfidf_matrix.npz')
print('  bm25.pkl')
print('  corpus_index.json')

Artifacts saved to /kaggle/working/ir_artifacts/:
  tfidf_vectorizer.pkl
  tfidf_matrix.npz
  bm25.pkl
  corpus_index.json


## 9. Live Demo

In [10]:
def demo_ir(question, top_k=3):
    print(f'\nQuestion: {question}')
    print('=' * 60)
    results = retrieve_hybrid(question, top_k=top_k)
    for rank, (idx, score) in enumerate(results, 1):
        print(f'Rank {rank} | Score: {score:.4f}')
        print(corpus_passages[idx][:300])
        print('-' * 60)

demo_ir("What is the theory of relativity?")
demo_ir("Who wrote Romeo and Juliet?")
demo_ir("When did World War II end?")


Question: What is the theory of relativity?
Rank 1 | Score: 1.0000
In the classical case, the invariance, or symmetry, group and the covariance group coincide, but, interestingly enough, they part ways in relativistic physics. The symmetry group of the general theory of relativity includes all differentiable transformations, i.e., all properties of an object are dy
------------------------------------------------------------
Rank 2 | Score: 0.7796
Albert Einstein proposed that the laws of physics should be based on the principle of relativity. This principle holds that the rules of physics must be the same for all observers, regardless of the frame of reference that is used, and that light propagates at the same speed in all reference frames.
------------------------------------------------------------
Rank 3 | Score: 0.6267
Idealist notions took a strong hold among physicists of the early 20th century confronted with the paradoxes of quantum physics and the theory of relativity. In T